In [0]:
file_path = "/Volumes/dev_catalog/landing_zone/landing_vol/raw_roads/"
file_path2 = "/Volumes/dev_catalog/landing_zone/landing_vol/raw_traffic/"

In [0]:
df = spark.read.csv(file_path, header=True, inferSchema=True)
display(df)

In [0]:
%sql
CREATE OR REPLACE TABLE dev_catalog.bronze.raw_roads
LOCATION 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_roads'
AS
SELECT *, current_timestamp() AS ingestion_timestamp
from read_files('/Volumes/dev_catalog/landing_zone/landing_vol/raw_roads/', 
    format =>'csv',
    header => true,
    inferSchema => true
    
);

In [0]:
%sql
DESCRIBE TABLE EXTENDED dev_catalog.bronze.raw_roads;

In [0]:
df = spark.read.csv(path=file_path2, header=True, inferSchema=True)
df.display()
df.schema

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.bronze.raw_traffic(
    count_point_id STRING, year INT, region_id INT, region_name STRING, region_ons_code STRING, local_authority_id DOUBLE, local_authority_name STRING, local_authority_code STRING, road_name STRING, road_category STRING, road_type STRING, start_junction_road_name STRING, end_junction_road_name STRING, easting INT, northing INT, latitude DOUBLE, longitude DOUBLE, link_length_km STRING, link_length_miles STRING, estimation_method STRING, estimation_method_detailed STRING, direction_of_travel STRING, pedal_cycles INT, two_wheeled_motor_vehicles INT, cars_and_taxis INT, buses_and_coaches INT, LGVs INT, HGVs_2_rigid_axle INT, HGVs_3_rigid_axle INT, HGVs_4_or_more_rigid_axle INT, HGVs_3_or_4_articulated_axle INT, HGVs_5_articulated_axle INT, HGVs_6_articulated_axle INT, all_HGVs INT, all_motor_vehicles INT
)
LOCATION 'abfss://bronze@mamataustrafficstorage.dfs.core.windows.net/raw_traffic'

In [0]:
%sql
COPY INTO dev_catalog.bronze.raw_traffic
FROM '/Volumes/dev_catalog/landing_zone/landing_vol/raw_traffic/'
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true' , 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
CREATE OR REPLACE TABLE dev_catalog.bronze.vehicle_type_lookup (
  vehicle_code STRING,
  vehicle_label STRING
);
INSERT INTO dev_catalog.bronze.vehicle_type_lookup VALUES
  ('EV_Car', 'Electric Car'), ('EV_Bike', 'Electric Bike'),
  ('LGV_Type', 'Light Goods Vehicle'), ('HGV_Type', 'Heavy Goods Vehicle');